# 07 -- Static Trajectory Planning

Build a deterministic lunar terrain configuration space, compute a travel-time
field and minimum-time path, and inspect both as maps. This notebook is based on
`examples/32_static_trajectory.py` and uses only public `ls.trajectory` APIs.

## Setup

In [ ]:
import sys, os
from pathlib import Path

def _repo_root():
    """Find the Lunarscout repository root from the kernel working directory."""
    for start in [Path.cwd()] + list(Path.cwd().parents):
        if (start / "src" / "lunarscout" / "__init__.py").exists():
            return start
    raise RuntimeError(
        "Cannot locate Lunarscout repository root. "
        "Launch Jupyter from the repository root directory."
    )

_REPO = _repo_root()
sys.path.insert(0, str(_REPO / "src"))
sys.path.insert(0, str(_REPO / "examples"))


import lunarscout as ls
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

%matplotlib inline

from _example_support import synthetic_georef

georef = synthetic_georef(width=12, height=9, pixel_size=10.0, nodata=None)
rows, columns = np.indices((georef.height, georef.width), dtype=np.float64)
elevation_m = 100.0 + 0.25 * columns + 0.1 * rows
traversable = np.ones(elevation_m.shape, dtype=bool)
traversable[1:8, 5] = False
traversable[6, 5] = True
valid = np.ones(elevation_m.shape, dtype=bool)
valid[0, 10:] = False
start, goal = (1, 1), (10, 7)
configuration = traversable & valid

## Configuration space

Terrain elevation is shown beneath the cells removed by validity and
traversability constraints. The opening in the vertical barrier is the only
crossing from the left side to the goal.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
image = ax.imshow(elevation_m, cmap="terrain", origin="upper")
blocked = np.ma.masked_where(configuration, ~configuration)
ax.imshow(blocked, cmap=ListedColormap(["black"]), alpha=0.75, origin="upper")
ax.scatter(*start, marker="o", s=100, color="cyan", edgecolor="black", label="start")
ax.scatter(*goal, marker="*", s=180, color="gold", edgecolor="black", label="goal")
ax.set(title="Static configuration space", xlabel="x (cell)", ylabel="y (cell)")
ax.legend(handles=ax.get_legend_handles_labels()[0] + [Patch(color="black", label="blocked")])
fig.colorbar(image, ax=ax, label="elevation (m)")
fig.tight_layout()
plt.show()

## Travel model and planning

In [ ]:
slip = ls.trajectory.SlipFunction(
    signed_slopes=(-0.5, 0.0, 0.5),
    factors=(0.8, 1.0, 2.0),
    extrapolation="infeasible",
)
model = ls.trajectory.StaticTravelModel(
    speed_m_per_h=36.0,
    include_diagonals=True,
    slip=slip,
)
field = ls.trajectory.static_travel_time(
    traversable, georef, start,
    valid=valid, elevation=elevation_m, model=model,
)
route = ls.trajectory.static_path(
    traversable, georef, start, goal,
    valid=valid, elevation=elevation_m, model=model,
)
print(f"Reachable: {route.reachable}")
print(f"Travel time: {route.travel_time_hours:.3f} hours")
print(f"Path cells: {len(route.path)}")

## Travel-time field and path overlay

In [ ]:
times = np.where(field.reached, field.travel_time_hours, np.nan)
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

travel_image = axes[0].imshow(times, cmap="viridis", origin="upper")
axes[0].set_title("Minimum travel time from start")
fig.colorbar(travel_image, ax=axes[0], label="hours")

terrain_image = axes[1].imshow(elevation_m, cmap="terrain", origin="upper")
axes[1].imshow(blocked, cmap=ListedColormap(["black"]), alpha=0.65, origin="upper")
axes[1].set_title("Minimum-time path over terrain")
fig.colorbar(terrain_image, ax=axes[1], label="elevation (m)")

for ax in axes:
    ax.plot(route.path[:, 0], route.path[:, 1], "w-o", lw=2, ms=4,
            markeredgecolor="black")
    ax.scatter(*start, color="cyan", edgecolor="black", s=90, zorder=4)
    ax.scatter(*goal, color="gold", edgecolor="black", marker="*", s=170, zorder=4)
    ax.set(xlabel="x (cell)", ylabel="y (cell)")
plt.show()

## Unreachable comparison

Close the barrier opening and compare the resulting configuration space. A
valid but unreachable goal returns an ordinary result rather than an exception.

In [ ]:
closed = traversable.copy()
closed[:, 5] = False
unreachable = ls.trajectory.static_path(
    closed, georef, start, goal,
    valid=valid, elevation=elevation_m, model=model,
)
fig, ax = plt.subplots(figsize=(9, 4))
ax.imshow(closed & valid, cmap=ListedColormap(["black", "white"]), origin="upper")
ax.scatter(*start, color="cyan", edgecolor="black", s=90)
ax.scatter(*goal, color="gold", edgecolor="black", marker="*", s=170)
ax.set(title=f"Closed barrier -- reachable: {unreachable.reachable}",
       xlabel="x (cell)", ylabel="y (cell)")
plt.show()